# BSDS500 Pixel Feature Extraction

Build a tabular pixel-level feature dataset from the BSDS500 image segmentation dataset.

**Task:** binary "edge pixel vs non-edge pixel" classification.

For every sampled pixel we compute simple colour / gradient / texture / position features,
and label it `1` (edge) if a majority of the human annotators marked it as a segment
boundary, else `0`.

**Output:** `data/bsds_features.csv` — used by every question in this lab.

> This version auto-discovers your `images` and ground-truth folders wherever they are
> under the notebook's working directory, since BSDS500 downloads come in a few different
> layouts (`archive/images/train` + `archive/ground_truth/train`, `BSR/BSDS500/data/images/train`
> + `.../groundTruth/train`, a flat `images/` + `groundTruth/` with no `archive/` wrapper, etc.).
> No manual path editing needed unless discovery fails.

## 0. Imports and auto-discovery of the dataset folders

In [ ]:
import numpy as np
import pandas as pd
from PIL import Image
from scipy import ndimage
import scipy.io as sio
from pathlib import Path

RNG = np.random.default_rng(42)
ROOT = Path.cwd().resolve()
OUT_CSV = ROOT / "data" / "bsds_features.csv"

SAMPLES_PER_CLASS_PER_IMAGE = 60   # -> up to 120 rows / image
BOUNDARY_VOTE_THRESHOLD = 0.5      # fraction of annotators that must agree

print(f"ROOT = {ROOT}")

In [ ]:
def find_dataset_dirs(root, search_depth=6):
    """
    Search under `root` for a folder of .jpg images and a folder of .mat
    ground-truth files whose filenames overlap (same BSDS image ids), trying
    the common BSDS500 layouts and folder-name variants.
    """
    # Any directory (up to search_depth levels deep) that directly contains .jpg files
    jpg_dirs = set()
    mat_dirs = set()
    for p in root.rglob("*"):
        try:
            depth = len(p.relative_to(root).parts)
        except ValueError:
            continue
        if depth > search_depth or not p.is_dir():
            continue
        if any(f.suffix.lower() == ".jpg" for f in p.iterdir() if f.is_file()):
            jpg_dirs.add(p)
        if any(f.suffix.lower() == ".mat" for f in p.iterdir() if f.is_file()):
            mat_dirs.add(p)

    best_pair, best_overlap = None, 0
    for jd in jpg_dirs:
        jpg_ids = {f.stem for f in jd.glob("*.jpg")}
        for md in mat_dirs:
            mat_ids = {f.stem for f in md.glob("*.mat")}
            overlap = len(jpg_ids & mat_ids)
            if overlap > best_overlap:
                best_overlap = overlap
                best_pair = (jd, md)

    return best_pair, best_overlap, jpg_dirs, mat_dirs


best_pair, best_overlap, jpg_dirs, mat_dirs = find_dataset_dirs(ROOT)

if best_pair is None or best_overlap == 0:
    print("Could not auto-discover matching image / ground-truth folders under:")
    print(f"  {ROOT}\n")
    print(f"Folders containing .jpg files found: {len(jpg_dirs)}")
    for d in sorted(jpg_dirs):
        print(f"  - {d}")
    print(f"\nFolders containing .mat files found: {len(mat_dirs)}")
    for d in sorted(mat_dirs):
        print(f"  - {d}")
    raise FileNotFoundError(
        "\nNo folder pair with matching filenames (e.g. 100075.jpg <-> 100075.mat) was found.\n"
        "This usually means the BSDS500 dataset hasn't been extracted next to this notebook yet, "
        "or it's nested deeper than 6 levels.\n"
        "Fix: download/extract BSDS500 so this notebook's working directory contains it "
        "somewhere, OR set IMG_DIR / GT_DIR manually below to the correct paths, e.g.:\n"
        "  IMG_DIR = ROOT / \'archive\' / \'images\' / \'train\'\n"
        "  GT_DIR  = ROOT / \'archive\' / \'ground_truth\' / \'train\'"
    )

IMG_DIR, GT_DIR = best_pair
print(f"Auto-discovered dataset folders (best match: {best_overlap} overlapping ids)")


### Confirm what was found

In [ ]:
n_imgs = len(list(IMG_DIR.glob("*.jpg")))
n_mats = len(list(GT_DIR.glob("*.mat")))
print(f"IMG_DIR = {IMG_DIR}")
print(f"  -> {n_imgs} .jpg files")
print(f"GT_DIR  = {GT_DIR}")
print(f"  -> {n_mats} .mat files")
print(f"Matching ids (used for extraction) = {best_overlap}")

if best_overlap < min(n_imgs, n_mats):
    print(f"\nNote: {min(n_imgs, n_mats) - best_overlap} files don't have a matching "
          f"counterpart by filename and will be skipped during extraction — this is normal "
          f"if IMG_DIR/GT_DIR include val/test images alongside train.")

### If auto-discovery picked the wrong folder pair

If your dataset has multiple splits (train/val/test) sitting in sibling folders, the
search above always picks the pair with the **most matching filenames**, which is usually
train (has the most images). If you specifically want a different split, override
`IMG_DIR` / `GT_DIR` manually in the cell below (uncomment and edit), then re-run the
"clean report" cell above to confirm.

In [ ]:
# IMG_DIR = ROOT / "archive" / "images" / "train"
# GT_DIR = ROOT / "archive" / "ground_truth" / "train"

## 1. Helper functions

In [ ]:
def consensus_boundary(mat_path):
    d = sio.loadmat(mat_path)
    gt = d["groundTruth"]
    n = gt.shape[1]
    acc = None
    for i in range(n):
        b = gt[0, i][0, 0]["Boundaries"].astype(np.float32)
        acc = b if acc is None else acc + b
    return (acc / n) >= BOUNDARY_VOTE_THRESHOLD


def extract_features(gray, rgb):
    sobel_x = ndimage.sobel(gray, axis=1)
    sobel_y = ndimage.sobel(gray, axis=0)
    grad_mag = np.hypot(sobel_x, sobel_y)
    grad_mag = grad_mag / (grad_mag.max() + 1e-8)
    grad_dir = np.arctan2(sobel_y, sobel_x)

    laplacian = np.abs(ndimage.laplace(gray))

    local_mean = ndimage.uniform_filter(gray, size=5)
    local_sqmean = ndimage.uniform_filter(gray ** 2, size=5)
    local_std = np.sqrt(np.clip(local_sqmean - local_mean ** 2, 0, None))

    return {
        "R": rgb[..., 0],
        "G": rgb[..., 1],
        "B": rgb[..., 2],
        "gray": gray,
        "grad_mag": grad_mag,
        "grad_dir": grad_dir,
        "laplacian": laplacian,
        "local_std": local_std,
    }

## 2. Main extraction loop

Tracks *why* each file was skipped and raises a clear error if nothing was collected,
instead of writing an empty CSV and crashing on the next cell.

In [ ]:
def main():
    mat_files = sorted(GT_DIR.glob("*.mat"))
    rows = []
    skip_counts = {"missing_jpg": 0, "shape_mismatch": 0, "no_valid_pixels": 0}

    for k, mat_path in enumerate(mat_files):
        img_path = IMG_DIR / (mat_path.stem + ".jpg")
        if not img_path.exists():
            skip_counts["missing_jpg"] += 1
            continue

        img = Image.open(img_path).convert("RGB")
        rgb = np.asarray(img, dtype=np.float32) / 255.0
        gray = rgb.mean(axis=2)

        edge_map = consensus_boundary(mat_path)
        # BSDS images can be landscape or portrait; ground truth matches orientation.
        if edge_map.shape != gray.shape:
            skip_counts["shape_mismatch"] += 1
            continue

        feats = extract_features(gray, rgb)
        h, w = gray.shape

        yy, xx = np.mgrid[0:h, 0:w]
        x_norm = xx / (w - 1)
        y_norm = yy / (h - 1)

        # avoid a 4px border so the filters above are well defined
        border = 4
        valid = np.zeros_like(edge_map, dtype=bool)
        valid[border:-border, border:-border] = True

        edge_idx = np.argwhere(edge_map & valid)
        nonedge_idx = np.argwhere((~edge_map) & valid)

        n_edge = min(SAMPLES_PER_CLASS_PER_IMAGE, len(edge_idx))
        if n_edge == 0:
            skip_counts["no_valid_pixels"] += 1
            continue
        n_nonedge = min(SAMPLES_PER_CLASS_PER_IMAGE, len(nonedge_idx))

        pick_edge = edge_idx[RNG.choice(len(edge_idx), n_edge, replace=False)]
        pick_nonedge = nonedge_idx[RNG.choice(len(nonedge_idx), n_nonedge, replace=False)]
        picks = np.vstack([pick_edge, pick_nonedge])
        labels = np.array([1] * n_edge + [0] * n_nonedge)

        for (r, c), lab in zip(picks, labels):
            rows.append({
                "image_id": mat_path.stem,
                "R": feats["R"][r, c],
                "G": feats["G"][r, c],
                "B": feats["B"][r, c],
                "gray": feats["gray"][r, c],
                "grad_mag": feats["grad_mag"][r, c],
                "grad_dir": feats["grad_dir"][r, c],
                "laplacian": feats["laplacian"][r, c],
                "local_std": feats["local_std"][r, c],
                "x_norm": x_norm[r, c],
                "y_norm": y_norm[r, c],
                "is_edge": lab,
            })

        if (k + 1) % 40 == 0:
            print(f"  processed {k + 1}/{len(mat_files)} images, rows so far={len(rows)}")

    if not rows:
        raise RuntimeError(
            f"No rows were collected from {len(mat_files)} ground-truth files. "
            f"Skip reasons: {skip_counts}. "
            "This almost always means IMG_DIR/GT_DIR point at the wrong folders, "
            "or the .jpg filenames don't match the .mat filenames (both should share "
            "the same 6-digit BSDS image id, e.g. 100075.jpg / 100075.mat)."
        )

    df = pd.DataFrame(rows)
    OUT_CSV.parent.mkdir(exist_ok=True, parents=True)
    df.to_csv(OUT_CSV, index=False)
    print(f"Saved {len(df)} rows x {df.shape[1]} cols -> {OUT_CSV}")
    print(f"Skipped: {skip_counts}")
    print(df["is_edge"].value_counts())
    return df

In [ ]:
df = main()
df.head()

---
# Shared helpers (`common.py` equivalent)

Used by every `q*.py` / question notebook — keeps the train/test split and feature
columns identical across all 7 questions, since they all reuse the same
BSDS500-derived dataset (`data/bsds_features.csv`).

In [ ]:
import json
from sklearn.model_selection import train_test_split

FIG_DIR = ROOT / "results" / "figures"
METRIC_DIR = ROOT / "results" / "metrics"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METRIC_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLS = ["R", "G", "B", "gray", "grad_mag", "grad_dir",
                "laplacian", "local_std", "x_norm", "y_norm"]
LABEL_COL = "is_edge"
RANDOM_STATE = 42


def load_split(test_size=0.2, scale=True):
    if not OUT_CSV.exists():
        raise FileNotFoundError(
            f"{OUT_CSV} does not exist yet — run the feature-extraction cells above first."
        )
    data = pd.read_csv(OUT_CSV)
    X = data[FEATURE_COLS].values.astype(np.float64)
    y = data[LABEL_COL].values.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y
    )

    if scale:
        mu, sigma = X_train.mean(axis=0), X_train.std(axis=0)
        sigma[sigma == 0] = 1.0
        X_train = (X_train - mu) / sigma
        X_test = (X_test - mu) / sigma

    return X_train, X_test, y_train, y_test


def save_metrics(name, d):
    path = METRIC_DIR / f"{name}.json"
    with open(path, "w") as f:
        json.dump(d, f, indent=2, default=float)
    print(f"saved metrics -> {path}")

In [ ]:
# quick sanity check
X_train, X_test, y_train, y_test = load_split()
X_train.shape, X_test.shape, y_train.mean(), y_test.mean()